# Push 2 Chord Drone

Chromatic keyboard + note toggle + chord bank on Push 2.

- **Rows 2-7**: Keyboard (live play) or Note Toggle (chord building), toggled via **Record** button
- **Row 0**: Chord bank — press to load, **Convert + press** to save
- **Row 1**: Free for live-coding experiments
- **Arrows / OctaveUp/Down**: Shift the grid

In [ ]:
import { Push2 } from "@/push2/push2.ts"
import { MidiAccess } from "@/midi/mod.ts"
import { GridLayout } from "@/push2/grid_layout.ts"
import { KeyboardModule } from "@/push2/modules/keyboard.ts"
import { NoteToggleModule } from "@/push2/modules/note_toggle.ts"
import { BankModule } from "@/push2/modules/bank.ts"

const OUTPUT_NAME = 'IAC Driver Bus 1'

const midi = MidiAccess.open()
const outputs = midi.listOutputs()
console.log('MIDI outputs:', outputs.map(p => p.name))

const outputInfo = outputs.find(p => p.name.includes(OUTPUT_NAME))!
const iacOut = midi.openOutput(outputInfo.id)

const push = Push2.create()
console.log('Push 2 connected, IAC output:', outputInfo.name)

In [ ]:
import { COLOR } from "@/push2/constants.ts"

const CH = 0
const layout = new GridLayout({ rows: [2, 7], baseNote: 36, rowInterval: 5 })

// Drone notes tracked for keyboard guard
const droneNotes = new Set<number>()

const noteToggle = new NoteToggleModule(push, layout, {
  noteOn: (note, vel) => iacOut.noteOn(CH, note, vel),
  noteOff: (note) => iacOut.noteOff(CH, note, 0),
})

// Sync drone state + keyboard highlights
noteToggle.onChange((notes) => {
  droneNotes.clear()
  for (const note of notes.keys()) droneNotes.add(note)
  const hl = new Map<number, number>()
  for (const note of notes.keys()) hl.set(note, COLOR.YELLOW)
  keyboard.setHighlights(hl)
})

const keyboard = new KeyboardModule(push, layout, {
  noteOn: (note, vel) => iacOut.noteOn(CH, note, vel),
  noteOff: (note) => { if (!droneNotes.has(note)) iacOut.noteOff(CH, note, 0) },
})

const bank = new BankModule<Map<number, number>>(
  push,
  (_index) => noteToggle.getOnNotes(),
  (chord) => {
    noteToggle.setNotes(chord)
    noteToggle.playAllOnNotes(true)
  },
  {
    rows: [0, 0],
    offFn: () => noteToggle.allNotesOff(),
  },
)

// Start with keyboard active
keyboard.activate()
bank.activate()
let keyboardActive = true

push.onButtonPressed("Record", () => {
  if (keyboardActive) {
    keyboard.deactivate()
    noteToggle.activate()
  } else {
    noteToggle.deactivate()
    keyboard.activate()
  }
  keyboardActive = !keyboardActive
  console.log(keyboardActive ? 'Keyboard mode' : 'Note toggle mode')
})

console.log('Modules active. Record = toggle keyboard/note-toggle')

## Free Row (row 1)

Row 1 (i=1) is unassigned. Use this cell to experiment — e.g. map row 1 pads to trigger inversions, scene changes, etc. `push`, `iacOut`, `noteToggle`, `bank`, `layout` are all in scope.

In [ ]:
import { padIJToN, COLOR } from "@/push2/constants.ts"

// Store unsubs so re-running this cell replaces handlers instead of stacking
const _row1Unsubs: (() => void)[] = (globalThis as any)._row1Unsubs ?? []
_row1Unsubs.forEach(fn => fn())

const row1Unsubs: (() => void)[] = [];
(globalThis as any)._row1Unsubs = row1Unsubs

// Example: row 1 pads as 8 quick-trigger buttons
// Re-run this cell to redefine behavior on the fly
row1Unsubs.push(push.onPadPressed((_padN, [i, j], _vel) => {
  if (i !== 1) return
  console.log(`Row 1 pad [${i},${j}] pressed`)
  push.setPadColor(padIJToN(i, j), COLOR.RED)
}))
row1Unsubs.push(push.onPadReleased((_padN, [i, j]) => {
  if (i !== 1) return
  push.setPadColor(padIJToN(i, j), COLOR.BLACK)
}))

In [ ]:
// Cleanup: silence notes, clear LEDs, close MIDI
noteToggle.allNotesOff()
keyboard.deactivate()
noteToggle.deactivate()
bank.deactivate()
// Clear row 1
for (let j = 0; j < 8; j++) push.setPadColor(padIJToN(1, j), COLOR.BLACK)

push.close()
iacOut.close()
midi.close()
console.log('Cleaned up')